<a href="https://colab.research.google.com/github/ard714/Deepflow_stamatics/blob/main/deepflow2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dog Breed Identification using Transfer Learning
# Major Assignment 2 - CNN with Transfer Learning

# Install required packages
!pip install tensorflow==2.13.0
!pip install keras==2.13.1
!pip install pillow
!pip install matplotlib
!pip install seaborn
!pip install scikit-learn

# Import necessary libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import os
import cv2
from PIL import Image
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Mount Google Drive to save models and results
from google.colab import drive
drive.mount('/content/drive')

# Download and extract the dataset
# You need to download the dataset from Kaggle first
# For this example, we'll assume the data is in /content/dog-breed-identification/

# Create directories if they don't exist
os.makedirs('/content/dog-breed-identification', exist_ok=True)

# Load the dataset
print("Loading dataset...")
train_df = pd.read_csv('/content/dog-breed-identification/labels.csv')
sample_submission = pd.read_csv('/content/dog-breed-identification/sample_submission.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Sample submission shape: {sample_submission.shape}")
print(f"Number of unique breeds: {train_df['breed'].nunique()}")

# Display basic statistics
print("\nDataset Overview:")
print(train_df.head())
print(f"\nBreed distribution (top 10):")
print(train_df['breed'].value_counts().head(10))

# Visualize breed distribution
plt.figure(figsize=(15, 8))
breed_counts = train_df['breed'].value_counts()
plt.subplot(1, 2, 1)
breed_counts.head(20).plot(kind='bar')
plt.title('Top 20 Dog Breeds Distribution')
plt.xlabel('Breed')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.hist(breed_counts.values, bins=30, alpha=0.7)
plt.title('Distribution of Breed Counts')
plt.xlabel('Number of Images per Breed')
plt.ylabel('Number of Breeds')
plt.tight_layout()
plt.show()

# Configuration parameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 0.001
NUM_CLASSES = train_df['breed'].nunique()

print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

# Create label mapping
breed_to_idx = {breed: idx for idx, breed in enumerate(train_df['breed'].unique())}
idx_to_breed = {idx: breed for breed, idx in breed_to_idx.items()}

# Add numeric labels to dataframe
train_df['label'] = train_df['breed'].map(breed_to_idx)

# Split data into train and validation sets
train_data, val_data = train_test_split(train_df, test_size=0.2, stratify=train_df['breed'], random_state=42)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Data preprocessing and augmentation
def preprocess_image(image_path, label, is_training=True):
    """Load and preprocess image"""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0

    if is_training:
        # Data augmentation for training
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.1)
        image = tf.image.random_contrast(image, 0.9, 1.1)
        image = tf.image.random_saturation(image, 0.9, 1.1)

    return image, label

# Create datasets
def create_dataset(dataframe, is_training=True):
    """Create TensorFlow dataset from dataframe"""
    image_paths = [f'/content/dog-breed-identification/train/{img_id}.jpg' for img_id in dataframe['id'].values]
    labels = dataframe['label'].values

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(lambda x, y: preprocess_image(x, y, is_training), num_parallel_calls=tf.data.AUTOTUNE)

    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Create train and validation datasets
train_dataset = create_dataset(train_data, is_training=True)
val_dataset = create_dataset(val_data, is_training=False)

# Visualize sample images
def visualize_samples(dataset, num_samples=8):
    """Visualize sample images from dataset"""
    plt.figure(figsize=(15, 10))

    for images, labels in dataset.take(1):
        for i in range(min(num_samples, len(images))):
            plt.subplot(2, 4, i + 1)
            plt.imshow(images[i])
            breed_name = idx_to_breed[labels[i].numpy()]
            plt.title(f'Breed: {breed_name}')
            plt.axis('off')

    plt.tight_layout()
    plt.show()

print("Sample training images:")
visualize_samples(train_dataset)

# Model Architecture using Transfer Learning
def create_model(num_classes, model_name='efficientnet'):
    """Create CNN model with transfer learning"""

    if model_name == 'efficientnet':
        base_model = EfficientNetB0(
            weights='imagenet',
            include_top=False,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
    elif model_name == 'resnet':
        base_model = ResNet50V2(
            weights='imagenet',
            include_top=False,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )

    # Freeze base model initially
    base_model.trainable = False

    # Add custom classification head
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    return model, base_model

# Create the model
model, base_model = create_model(NUM_CLASSES, model_name='efficientnet')

# Compile model
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', 'top_5_accuracy']
)

# Model summary
print("Model Architecture:")
model.summary()

# Define callbacks
callbacks = [
    ModelCheckpoint(
        '/content/drive/MyDrive/dog_breed_model_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# Train the model (Phase 1: Frozen base model)
print("Phase 1: Training with frozen base model...")
history_phase1 = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

# Fine-tuning (Phase 2: Unfreeze some layers)
print("\nPhase 2: Fine-tuning with unfrozen layers...")

# Unfreeze the base model
base_model.trainable = True

# Fine-tune from this layer onwards
fine_tune_at = len(base_model.layers) // 2

# Freeze all the layers before fine_tune_at
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE/10),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', 'top_5_accuracy']
)

# Continue training
history_phase2 = model.fit(
    train_dataset,
    epochs=EPOCHS-10,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1,
    initial_epoch=10
)

# Combine training histories
history = {}
for key in history_phase1.history.keys():
    history[key] = history_phase1.history[key] + history_phase2.history[key]

# Plot training history
def plot_training_history(history):
    """Plot training and validation metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Accuracy
    axes[0, 0].plot(history['accuracy'], label='Training Accuracy')
    axes[0, 0].plot(history['val_accuracy'], label='Validation Accuracy')
    axes[0, 0].set_title('Model Accuracy')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Loss
    axes[0, 1].plot(history['loss'], label='Training Loss')
    axes[0, 1].plot(history['val_loss'], label='Validation Loss')
    axes[0, 1].set_title('Model Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Top-5 Accuracy
    axes[1, 0].plot(history['top_5_accuracy'], label='Training Top-5 Accuracy')
    axes[1, 0].plot(history['val_top_5_accuracy'], label='Validation Top-5 Accuracy')
    axes[1, 0].set_title('Model Top-5 Accuracy')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Top-5 Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # Learning Rate (if available)
    if 'lr' in history:
        axes[1, 1].plot(history['lr'], label='Learning Rate')
        axes[1, 1].set_title('Learning Rate')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    else:
        axes[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

plot_training_history(history)

# Evaluate the model
print("Evaluating model on validation set...")
val_loss, val_accuracy, val_top5_accuracy = model.evaluate(val_dataset, verbose=0)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Top-5 Accuracy: {val_top5_accuracy:.4f}")

# Make predictions on validation set for detailed analysis
print("Making predictions on validation set...")
val_predictions = model.predict(val_dataset)
val_pred_classes = np.argmax(val_predictions, axis=1)

# Get true labels
val_true_labels = []
for _, labels in val_dataset:
    val_true_labels.extend(labels.numpy())
val_true_labels = np.array(val_true_labels)

# Classification report
print("\nClassification Report (Top 10 classes):")
top_10_breeds = train_df['breed'].value_counts().head(10).index.tolist()
top_10_indices = [breed_to_idx[breed] for breed in top_10_breeds]

# Filter predictions and true labels for top 10 classes
mask = np.isin(val_true_labels, top_10_indices)
filtered_true = val_true_labels[mask]
filtered_pred = val_pred_classes[mask]

if len(filtered_true) > 0:
    print(classification_report(filtered_true, filtered_pred,
                              target_names=[idx_to_breed[i] for i in top_10_indices]))

# Save the final model
model.save('/content/drive/MyDrive/cnn_model.h5')
print("Model saved successfully!")

# Prepare test data for predictions
def prepare_test_data():
    """Prepare test dataset for predictions"""
    test_dir = '/content/dog-breed-identification/test'
    test_images = []
    test_ids = []

    for filename in os.listdir(test_dir):
        if filename.endswith('.jpg'):
            test_ids.append(filename[:-4])  # Remove .jpg extension
            image_path = os.path.join(test_dir, filename)

            # Load and preprocess image
            image = tf.io.read_file(image_path)
            image = tf.image.decode_jpeg(image, channels=3)
            image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
            image = tf.cast(image, tf.float32) / 255.0
            test_images.append(image)

    return np.array(test_images), test_ids

# Load test data and make predictions
print("Loading test data...")
test_images, test_ids = prepare_test_data()

print(f"Number of test images: {len(test_images)}")

# Make predictions on test set
print("Making predictions on test set...")
test_predictions = model.predict(test_images, batch_size=BATCH_SIZE, verbose=1)

# Create submission file
submission_df = pd.DataFrame()
submission_df['id'] = test_ids

# Add probability columns for each breed
breed_columns = sorted(train_df['breed'].unique())
for i, breed in enumerate(breed_columns):
    breed_idx = breed_to_idx[breed]
    submission_df[breed] = test_predictions[:, breed_idx]

# Save submission file
submission_df.to_csv('/content/drive/MyDrive/dog_breed_predictions.csv', index=False)
print("Predictions saved successfully!")

# Display sample predictions
print("\nSample predictions:")
print(submission_df.head())

# Visualize some test predictions
def visualize_test_predictions(images, predictions, ids, num_samples=8):
    """Visualize test predictions"""
    plt.figure(figsize=(15, 10))

    for i in range(min(num_samples, len(images))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i])

        # Get top prediction
        top_pred_idx = np.argmax(predictions[i])
        top_breed = idx_to_breed[top_pred_idx]
        confidence = predictions[i][top_pred_idx]

        plt.title(f'ID: {ids[i]}\nPredicted: {top_breed}\nConfidence: {confidence:.3f}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

print("Sample test predictions:")
visualize_test_predictions(test_images, test_predictions, test_ids)

# Model performance summary
print("\n" + "="*50)
print("MODEL PERFORMANCE SUMMARY")
print("="*50)
print(f"Architecture: EfficientNetB0 with Transfer Learning")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Total Epochs: {EPOCHS}")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Final Validation Accuracy: {val_accuracy:.4f}")
print(f"Final Validation Top-5 Accuracy: {val_top5_accuracy:.4f}")
print(f"Final Validation Loss: {val_loss:.4f}")
print("="*50)

# Save training history
import pickle
with open('/content/drive/MyDrive/training_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print("Training history saved!")
print("All files saved to Google Drive!")